# Concepts

This page introduces the two core building blocks that the rest of the user guide refers to: **change detectors** and **interval scorers**. A change detector is the object you use for detecting changes, while an interval scorer is the user-specified component that tells the detector *what feature of the data to look for changes in*. Almost every detector in Skchange is composed of (1) a search algorithm, (2) an interval scorer, and (3) a penalty, and most of the flexibility of the library comes from being able to mix and match these three.

Read this page once before the task-specific guides. It is meant to give you the vocabulary; the *when to use which* questions are answered in the [Change Detection](./change_detection/index.rst) and [Segment Anomaly Detection](./anomaly_detection/index.rst) sections.


## Change detectors

A *change detector* is a scikit-learn-style estimator that finds changepoints in a single time series. All detectors in skchange inherit from `BaseChangeDetector`, and they all expose `predict_changepoints` and a derived `predict` returning per-sample segment labels. You initialise a detector with an interval scorer, a penalty, and any algorithm-specific parameters, then call `fit` and one of the `predict_*` methods.

Some detectors do more than that. Detectors that natively identify *anomalous segments* — contiguous intervals where the data deviates from a "normal" or "baseline" behaviour — additionally expose `predict_segment_anomalies`. For those detectors the segment anomalies are the primary output and the changepoints are derived from them. `CAPA` and `CircularBinarySegmentation` are the current examples; pure change detectors include `PELT`, `SeededBinarySegmentation`, and `MovingWindow`.

The next subsections describe the methods that make up the detector interface. The first three (`fit`, `predict`, `predict_changepoints`) are universal. The remaining ones are present when the detector natively produces that output.

### `fit(X, y=None)`

Fits the detector to data `X` and returns the fitted instance (`self`). `X` is a 2D numpy array of shape `(n_samples, n_features)`; univariate input is always 2D with `n_features=1`. `fit` validates parameters, stores any fitted attributes (suffixed with `_`), and may precompute quantities used by the `predict_*` methods.

### `predict(X)`

Returns a numpy array of segment labels of shape `(n_samples,)` — one integer label per input sample. This mirrors the output convention of sklearn clusterers and classifiers, so the array plugs directly into pipelines and metrics.

### `predict_changepoints(X)`

Returns a numpy array of changepoint indices of shape `(n_changepoints,)`. A changepoint marks the *inclusive start of a new segment*.

### `predict_segment_anomalies(X)`

Optional. Returns a numpy array of anomalous intervals of shape `(n_anomalies, 2)` with `[start, end)` rows. The samples outside these intervals are considered normal. Implemented by detectors that natively support segment-anomaly detection (e.g. `CAPA`, `CircularBinarySegmentation`).

### `predict_scores(X, return_index=False)`

Optional. Returns the detector's internal interval scoring objective as a 1D numpy array. The length depends on the algorithm — one entry per evaluated interval, candidate split point, etc. — and is not generally equal to `n_samples`. With `return_index=True` it returns a `(scores, index_dict)` tuple; `index_dict` carries algorithm-specific metadata that locates each score on the input timeline. Used by `skchange.new_api.tuning` for penalty calibration.

### `predict_all(X)`

Optional. Convenience method on detectors that compute several outputs in a single pass. Returns a dict whose keys are detector-specific (e.g. `{"changepoints": ..., "scores": ..., "affected_features": ...}`). Useful for power users who want every output without repeating work.


### Scikit-learn compatibility

All skchange detectors inherit from scikit-learn's `BaseEstimator` via the `BaseChangeDetector` class. This gives you sklearn-standard machinery for free, but a few sklearn tools are intentionally not supported because they assume properties that time series data do not have.

**What works:**

- **`get_params` / `set_params`** for inspecting and updating hyperparameters.
- **`sklearn.base.clone`** for making unfitted copies.
- **`Pipeline`** with most sklearn transformers.
- **Fitted-attribute convention**: attributes set in `fit` end with `_` (e.g. `detector.penalty_`), and `check_is_fitted` works as expected.

**What does not work, and why:**

- **`GridSearchCV` / `cross_val_score` and other CV utilities.** These tools concatenate fold rows into a single array before calling `predict`, which destroys series boundaries and shuffles time-ordered samples. Use the built-in tuning utilities in `skchange.new_api.tuning` instead, which are designed for time series detection tasks.

## Interval scorers

Interval scorers are the main components that make skchange modular, flexible and fast. They are responsible for evaluating a score over a large amount of data intervals. The choice of interval scorer represents the choice of distributional feature(s) to detect changes in — for example the mean, the variance, regression parameters, or the full distribution.

Apart from being constructed, interval scorers are not meant to be used directly. They are used internally by the detectors. To make full use of the library, however, it is important to understand how they work.

All interval scorers share a single base class, `BaseIntervalScorer`, and a single evaluation API:

- **`fit(X, y=None) -> self`** validates the data and stores any state needed across calls.
- **`precompute(X) -> cache`** returns any precomputed quantities (for example cumulative sums) needed to evaluate many intervals quickly. Detectors call this internally.
- **`evaluate(cache, interval_specs) -> np.ndarray`** evaluates the score on a batch of intervals in one call. The output is a 2D numpy array with one row per interval; the number of columns is either 1 or `n_features` depending on the scorer.

What distinguishes one scorer from another is the **`score_type` tag**, declared via the sklearn-style `__sklearn_tags__` mechanism. The tag tells the detector — and any composition adapter — what kind of score the object produces, and therefore what shape of `interval_specs` it expects:

| `score_type` | What it scores | `interval_specs` shape |
|---|---|---|
| `"cost"` | The cost/loss of a model fit to a single interval. | `(n, 2)` — start, end |
| `"change_score"` | The degree of change between two adjacent intervals. | `(n, 3)` — start, split, end |
| `"saving"` | The degree to which an interval deviates from a fixed baseline model. | `(n, 2)` — start, end |
| `"transient_score"` | The degree to which an interval is anomalous compared to its local context. | `(n, 4)` — start, split1, split2, end |

Detectors expose typed constructor arguments (`cost=`, `change_score=`, `saving=`, …) and reject scorers whose `score_type` tag does not match the slot. Helper predicates `is_cost`, `is_change_score`, `is_saving`, and `is_transient_score` are available in `skchange.new_api.interval_scorers` if you ever need to dispatch on the tag yourself.



To make each kind concrete, we use the following dataset throughout. It contains a change in both mean and variance at index 120.


In [ ]:
from skchange.new_api.datasets import generate_piecewise_normal_data

X = generate_piecewise_normal_data(
    means=[0, 5],
    variances=[16, 1],
    lengths=[120, 80],
    seed=101,
)


### Cost

A *cost* measures the cost/loss/error of a model fit to a data interval `X[s:e]`. For example, `L2Cost` returns the within-interval sum of squared deviations from the sample mean — small when the interval is well-modelled by a single mean, large when it straddles a change.


In [ ]:
from skchange.new_api.interval_scorers import L2Cost

cost = L2Cost().fit(X)
cache = cost.precompute(X)
cost.evaluate(cache, [[0, 10], [110, 130], [140, 160]])


The cost is much larger for the interval `[110, 130)` than for the other two, because that interval straddles the change point at index 120 and a single-mean model fits poorly there.

The computational bottleneck of most change-detection algorithms is to evaluate a score over a large number of intervals. In skchange this is solved by splitting work between `fit`/`precompute` (which prepares quantities such as cumulative sums) and `evaluate` (which uses them to score many intervals in a single, often `numba`-accelerated call).


### Change score

A *change score* takes in a start–split–end configuration `(s, k, e)` and measures the degree of change between the two adjacent intervals `X[s:k]` and `X[k:e]`. Change scores can be statistical tests, time-series distances, or any other measure of difference. A classical example is the CUSUM score for a change in mean.


In [ ]:
from skchange.new_api.interval_scorers import CUSUM

score = CUSUM().fit(X)
cache = score.precompute(X)
score.evaluate(cache, [[0, 5, 10], [110, 120, 130], [140, 150, 160]])


Again, the change score is largest for the interval that contains the change point at index 120.

One of the strengths of the interval-scorer abstraction is that **a cost can always be turned into a change score**. The `CostChangeScore` adapter does exactly this:

```python
from skchange.new_api.interval_scorers import CostChangeScore, L2Cost

change_score = CostChangeScore(L2Cost())
```

Internally, `CostChangeScore` evaluates

```python
change_score(s, k, e) = cost(s, e) - (cost(s, k) + cost(k, e))
```

which you can read as *"the cost of the interval without a change point minus the cost of the interval with a single change point"*.

Composition in skchange is **explicit** — you always pass the exact scorer kind the detector expects. A detector that takes `change_score=` will reject a bare cost; you wrap it in `CostChangeScore` yourself. This keeps detector `repr` and `get_params()` round-trippable: what you passed is exactly what runs.

The library also supports change scores that **cannot** be reduced to costs. This is different from libraries such as `ruptures`. There are quite a few important scores that can not be reduced to costs, including the Mann–Whitney U test, the Kolmogorov–Smirnov test, and scores for sparse change detection in high-dimensional data. Implementing these directly is also often more computationally efficient.


### Saving

A *saving* takes a start–end configuration `(s, e)` and measures the difference in cost between a locally estimated and a globally fixed model parameter over `X[s:e]`. The globally fixed parameter represents the "normal" data behaviour and is user-specified. In practice it should be estimated robustly from the data. Savings are the main type of anomaly score used in skchange because they are cheap to evaluate over many intervals.

Like change scores, **a cost can always be used to make a saving**.


In [ ]:
import numpy as np

from skchange.new_api.interval_scorers import L2Saving

baseline_mean = float(np.median(X[: X.shape[0] // 2]))
print(f"Baseline mean: {baseline_mean:.3f}")

saving = L2Saving(baseline_mean=baseline_mean).fit(X)
cache = saving.precompute(X)
saving.evaluate(cache, [[0, 10], [110, 130], [140, 160]])


The saving is largest for the last interval, whose mean differs most from the baseline.

### Transient score

A *transient score* (also known as a local anomaly score) receives a start–split1–split2–end configuration `(s, k1, k2, e)` and measures how anomalous the interval `X[k1:k2]` is compared to the local context to its left and right, `X[s:k1]` and `X[k2:e]`. In the literature, such anomalies are sometimes called epidemic changes. Like change scores, **a cost can always be turned into a transient score** via the `CostTransientScore` adapter. Transient scores are powerful but typically heavier to compute than savings, because there are many more start–split1–split2–end combinations than start–end ones.

## Composability: scorer + algorithm + penalty = detector

Most skchange detectors are built from three pieces:

1. An **interval scorer** that defines *what* feature to look for changes in.
2. A **search algorithm** that decides *which intervals* to evaluate the scorer on and *how* to compile the results into a final set of events.
3. A **penalty** that controls how strongly the detector resists declaring new changes/anomalies — the higher the penalty, the fewer events will be detected.

Detector constructors take a typed argument for each kind of scorer they accept:

| Detector | Scorer argument | Required `score_type` |
|---|---|---|
| `PELT` | `cost=` | `"cost"` |
| `SeededBinarySegmentation`, `MovingWindow`, `CircularBinarySegmentation` | `change_score=` | `"change_score"` |
| `CAPA` | `segment_saving=`, `point_saving=` | `"saving"` |

This is why a single cost can drive many detectors. The same `L2Cost` plugs directly into `PELT`, and — via `CostChangeScore(L2Cost())` — into `MovingWindow` and the binary-segmentation detectors. Conversely, a single algorithm can be steered to detect very different features by swapping the scorer. The trade-off is one extra construction step in exchange for maximum reuse, a small core API, and a `repr` that fully describes what runs.

## Custom interval scorers

It is straightforward to add a new interval scorer. Subclass `BaseIntervalScorer`, set the `score_type` tag to one of `"cost"`, `"change_score"`, `"saving"`, or `"transient_score"` via `__sklearn_tags__`, and implement `precompute` and `evaluate`. A fill-in template lives in the [`extension_templates`](https://github.com/NorskRegnesentral/skchange/tree/main/extension_templates) folder on GitHub. Once your scorer carries the right tag, it plugs into every detector that accepts that `score_type` — no detector changes needed.

